# BBO Capstone — Round 10 Query Generation
**Imperial Business School | Executive Master in ML/AI**  
**Candidate:** Gian Franco Cattaneo  
**Module:** 21.1 — Refining Strategies for Black-Box Optimisation  
**Date:** 2026-06-03  

---

## Pipeline Architecture
- **GP Kernel:** Matérn-5/2 (ARD) + WhiteKernel (noise) + ConstantKernel (amplitude)
- **Scaler:** StandardScaler on inputs (zero-mean, unit-variance)
- **Acquisition:** Expected Improvement (EI) — maximisation formulation
- **Optimiser:** L-BFGS-B with 35 random restarts (top-35 of 5,000 warm-start candidates)
- **Objective:** All 8 functions treated as maximisation targets
- **Data:** 9 rounds × 8 functions = 72 evaluated query points

In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND GLOBAL CONFIGURATION
# ============================================================
import numpy as np
import warnings
from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

# Global optimiser settings
N_RESTARTS    = 35      # L-BFGS-B restarts for acquisition optimisation
N_CANDIDATES  = 5000    # Random candidates for acquisition warm-start
DOMAIN_LOWER  = 0.0
DOMAIN_UPPER  = 0.999999

print('Libraries loaded successfully.')
print(f'  GP restarts       : {N_RESTARTS}')
print(f'  Acquisition cands : {N_CANDIDATES}')
print(f'  Domain            : [{DOMAIN_LOWER}, {DOMAIN_UPPER}]^d')

In [ ]:
# ============================================================
# CELL 2 — COMPLETE DATASET: ROUNDS 1–9
# ============================================================
# Each list entry is one round of queries.
# Index 0 = f1 (d=2), Index 7 = f8 (d=8).
# All functions: maximisation objective.

all_inputs = [
    # --- Round 1 (W12, 26/04/26) ---
    [np.array([0.034388, 0.909319]),
     np.array([0.695196, 0.395970]),
     np.array([0.548145, 0.174647, 0.303245]),
     np.array([0.440429, 0.425456, 0.378357, 0.397088]),
     np.array([0.000000, 0.675974, 0.999999, 0.999999]),
     np.array([0.464677, 0.242110, 0.574863, 0.999999, 0.000000]),
     np.array([0.000000, 0.241713, 0.327655, 0.218095, 0.375335, 0.747501]),
     np.array([0.064016, 0.008062, 0.123268, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 2 (W13, 06/04/26) ---
    [np.array([0.999999, 0.999999]),
     np.array([0.698486, 0.000000]),
     np.array([0.850892, 0.035316, 0.936193]),
     np.array([0.999999, 0.000000, 0.000000, 0.365908]),
     np.array([0.000000, 0.000000, 0.999999, 0.999999]),
     np.array([0.142734, 0.321812, 0.416483, 0.999999, 0.304415]),
     np.array([0.000000, 0.302741, 0.000000, 0.187177, 0.000000, 0.167183]),
     np.array([0.096074, 0.000000, 0.581701, 0.000000, 0.999999, 0.383890, 0.202188, 0.999999])],

    # --- Round 3 (W14, 08/04/26) ---
    [np.array([0.250000, 0.250000]),
     np.array([0.695000, 0.396000]),
     np.array([0.300000, 0.500000, 0.700000]),
     np.array([0.440000, 0.425000, 0.378000, 0.397000]),
     np.array([0.000000, 0.850000, 0.999999, 0.999999]),
     np.array([0.500000, 0.500000, 0.500000, 0.500000, 0.500000]),
     np.array([0.000000, 0.242000, 0.328000, 0.218000, 0.375000, 0.748000]),
     np.array([0.064000, 0.008000, 0.120000, 0.000000, 0.999999, 0.382000, 0.031000, 0.806000])],

    # --- Round 4 (W15, 19/04/26) ---
    [np.array([0.500000, 0.500000]),
     np.array([0.700000, 0.200000]),
     np.array([0.950000, 0.010000, 0.990000]),
     np.array([0.999999, 0.000000, 0.000000, 0.700000]),
     np.array([0.000000, 0.000000, 0.500000, 0.500000]),
     np.array([0.300000, 0.400000, 0.600000, 0.200000, 0.600000]),
     np.array([0.000000, 0.150000, 0.000000, 0.100000, 0.000000, 0.100000]),
     np.array([0.100000, 0.000000, 0.800000, 0.000000, 0.999999, 0.380000, 0.350000, 0.999999])],

    # --- Round 5 (W16, 23/04/26) ---
    [np.array([0.472781, 0.505546]),
     np.array([0.695211, 0.395970]),
     np.array([0.511275, 0.215264, 0.371049]),
     np.array([0.455000, 0.415000, 0.385000, 0.395000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.758817, 0.272673, 0.522143, 0.999999, 0.000000]),
     np.array([0.000000, 0.260000, 0.340000, 0.232000, 0.395000, 0.752000]),
     np.array([0.040000, 0.000000, 0.090000, 0.005000, 0.999999, 0.367013, 0.020000, 0.780000])],

    # --- Round 6 (W17, 01/05/26) ---
    [np.array([0.445562, 0.511092]),
     np.array([0.693000, 0.397000]),
     np.array([0.490000, 0.230000, 0.395000]),
     np.array([0.430000, 0.430000, 0.375000, 0.400000]),
     np.array([0.005000, 0.999999, 0.999999, 0.999999]),
     np.array([0.450000, 0.240000, 0.580000, 0.999999, 0.000000]),
     np.array([0.000000, 0.238000, 0.325000, 0.215000, 0.370000, 0.743000]),
     np.array([0.063000, 0.008000, 0.123000, 0.000000, 0.999999, 0.382000, 0.031000, 0.807000])],

    # --- Round 7 (W18, 06/05/26) ---
    [np.array([0.475000, 0.503000]),
     np.array([0.697000, 0.393000]),
     np.array([0.478000, 0.223000, 0.408000]),
     np.array([0.420000, 0.440000, 0.373000, 0.403000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.468000, 0.241000, 0.572000, 0.999999, 0.000000]),
     np.array([0.000000, 0.235000, 0.322000, 0.212000, 0.367000, 0.740000]),
     np.array([0.064016, 0.008062, 0.124000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 8 (W19, 18/05/26) ---
    [np.array([0.477000, 0.501000]),
     np.array([0.695000, 0.394000]),
     np.array([0.465000, 0.222000, 0.421000]),
     np.array([0.428000, 0.432000, 0.374000, 0.401000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.460000, 0.242000, 0.575000, 0.999999, 0.000000]),
     np.array([0.000000, 0.232000, 0.319000, 0.209000, 0.364000, 0.737000]),
     np.array([0.064016, 0.008062, 0.126000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 9 (W20, 25/05/26) ---
    [np.array([0.479000, 0.499000]),
     np.array([0.695200, 0.396500]),
     np.array([0.480000, 0.221000, 0.406000]),
     np.array([0.426000, 0.434000, 0.373000, 0.402000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.465000, 0.241000, 0.576000, 0.999999, 0.000000]),
     np.array([0.000000, 0.229000, 0.316000, 0.206000, 0.361000, 0.734000]),
     np.array([0.064016, 0.008062, 0.128000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],
]

all_outputs = [
    # Round 1
    [-2.4674747069022486e-270, 0.7237404632835625, -0.08911956876452833,
      0.25957575200735095, 2105.928152398213, -0.5507747202906804,
      2.207308607344047, 9.8595486103895],
    # Round 2
    [1.517648729565899e-192, 0.5297658866453171, -0.23982430098711077,
     -27.859767965401783, 1616.625747348229, -1.0045153236844038,
      0.050978228653516464, 9.2933769573024],
    # Round 3
    [9.797748409814019e-42, 0.5263661301012157, -0.1139602029925284,
      0.2748080020297299, 2932.694991178572, -1.0159268487405835,
      2.2071746109147172, 9.8591545999995],
    # Round 4
    [2.6752879910742468e-09, 0.5813540452269076, -0.4594065810473597,
     -30.894440825162423, 83.9625, -1.223884840915805,
      0.02363347322274405, 8.5129002799994],
    # Round 5
    [8.168635327996585e-08, 0.6238852457166373, -0.0707083820875107,
     -0.3996600230633507, 4440.480873479282, -0.9105784720492842,
      2.113257173327904, 9.8387496578305],
    # Round 6
    [-5.316626716773722e-07, 0.3979411317837673, -0.05294904589920826,
      0.4636173326649424, 4440.482959868813, -0.5765502837220897,
      2.2377743369228718, 9.8591202999995],
    # Round 7
    [1.3110732833867364e-07, 0.4872120229835646, -0.03529824725945802,
      0.36348229336051263, 4440.480873479282, -0.636071744816847,
      2.250193135816396, 9.8595765698615],
    # Round 8
    [1.74738012975983e-07, 0.5737744968898595, -0.04183377358963399,
      0.471059092282037, 4440.480873479282, -0.5620034889773333,
      2.2611593345633962, 9.8596365698615],
    # Round 9
    [2.2162534249618986e-07, 0.36665142548207275, -0.04723162936807271,
      0.4679683636993599, 4440.480873479282, -0.6061839383029959,
      2.2706472497897785, 9.8596725698615],
]

# Reorganise: per-function lists of (X, y)
N_FUNCTIONS = 8
N_ROUNDS    = len(all_inputs)

func_X = [[] for _ in range(N_FUNCTIONS)]
func_y = [[] for _ in range(N_FUNCTIONS)]

for rnd in range(N_ROUNDS):
    for fi in range(N_FUNCTIONS):
        func_X[fi].append(all_inputs[rnd][fi])
        func_y[fi].append(all_outputs[rnd][fi])

for fi in range(N_FUNCTIONS):
    func_X[fi] = np.vstack(func_X[fi])
    func_y[fi] = np.array(func_y[fi])

dims = [func_X[fi].shape[1] for fi in range(N_FUNCTIONS)]

print(f'Dataset loaded: {N_ROUNDS} rounds × {N_FUNCTIONS} functions = {N_ROUNDS*N_FUNCTIONS} evaluated points')
print(f'{"Func":>6} {"Dims":>6} {"N pts":>6} {"y_best":>14} {"x_best"}')
print('-' * 75)
for fi in range(N_FUNCTIONS):
    ib = np.argmax(func_y[fi])
    print(f'  f{fi+1:>2}  {dims[fi]:>5}  {len(func_y[fi]):>5}  '
          f'{func_y[fi].max():>14.6f}  {func_X[fi][ib]}')

In [ ]:
# ============================================================
# CELL 3 — GP-BO PIPELINE
# ============================================================

def build_kernel(d):
    """
    Construct the GP kernel:
      ConstantKernel (amplitude) × Matern-5/2 (ARD, one length-scale per dim)
      + WhiteKernel (homoscedastic noise)
    ARD enabled by passing length_scale as a d-vector.
    """
    return (ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=np.ones(d),
                     length_scale_bounds=(1e-3, 10.0),
                     nu=2.5)
            + WhiteKernel(noise_level=1e-4,
                          noise_level_bounds=(1e-8, 1e-1)))


def fit_gp(X_raw, y):
    """
    Scale inputs, fit GP, return (gp, scaler).
    Uses 10 L-BFGS-B restarts for kernel hyperparameter optimisation.
    """
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)
    d      = X_raw.shape[1]
    gp = GaussianProcessRegressor(
        kernel=build_kernel(d),
        n_restarts_optimizer=10,
        normalize_y=True,
        random_state=42
    )
    gp.fit(X_sc, y)
    return gp, scaler


def expected_improvement(x, gp, scaler, y_best, xi=0.01):
    """
    Expected Improvement for MAXIMISATION.

    EI(x) = (μ(x) - y_best - ξ) · Φ(Z) + σ(x) · φ(Z)
    Z      = (μ(x) - y_best - ξ) / σ(x)

    Returns negative EI (minimisation target for scipy.optimize).
    """
    x_sc       = scaler.transform(x.reshape(1, -1))
    mu, sigma  = gp.predict(x_sc, return_std=True)
    mu, sigma  = mu[0], max(sigma[0], 1e-9)
    improvement = mu - y_best - xi
    Z           = improvement / sigma
    ei          = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -max(ei, 0.0)   # negative for minimiser


def optimise_acquisition(gp, scaler, y_best, d, n_restarts=N_RESTARTS,
                          n_candidates=N_CANDIDATES):
    """
    Maximise EI over [0, 0.999999]^d via multi-start L-BFGS-B.

    Strategy:
      1. Evaluate EI at n_candidates random uniform samples.
      2. Select the top-n_restarts candidates as L-BFGS-B starting points.
      3. Return the argmax across all local optima found.
    """
    bounds = [(DOMAIN_LOWER, DOMAIN_UPPER)] * d

    X_cand  = np.random.uniform(DOMAIN_LOWER, DOMAIN_UPPER, size=(n_candidates, d))
    ei_cand = np.array([-expected_improvement(x, gp, scaler, y_best)
                         for x in X_cand])
    top_idx = np.argsort(ei_cand)[:n_restarts]
    x0_pool = X_cand[top_idx]

    best_x, best_ei = None, np.inf
    for x0 in x0_pool:
        res = minimize(
            expected_improvement, x0,
            args=(gp, scaler, y_best),
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 500, 'ftol': 1e-12}
        )
        if res.fun < best_ei:
            best_ei = res.fun
            best_x  = res.x

    return best_x, -best_ei   # return (x*, EI value)


print('GP-BO pipeline functions defined.')
print('  build_kernel()          — Matern-5/2 ARD + WhiteKernel')
print('  fit_gp()                — StandardScaler + GPR (10 HP restarts)')
print('  expected_improvement()  — EI (maximisation, ξ=0.01)')
print('  optimise_acquisition()  — Multi-start L-BFGS-B (35 restarts)')

In [ ]:
# ============================================================
# CELL 4 — RUN GP-BO FOR ALL 8 FUNCTIONS
# ============================================================

gp_suggestions = []
gp_ei_values   = []

print(f'{"Func":>6}  {"d":>3}  {"y_best":>12}  {"GP-EI x*":>55}  {"EI":>10}')
print('=' * 95)

for fi in range(N_FUNCTIONS):
    X      = func_X[fi]
    y      = func_y[fi]
    d      = dims[fi]
    y_best = np.max(y)

    gp, scaler = fit_gp(X, y)
    x_next, ei_val = optimise_acquisition(gp, scaler, y_best, d)

    gp_suggestions.append(x_next)
    gp_ei_values.append(ei_val)

    x_str = '  '.join([f'{v:.6f}' for v in x_next])
    print(f'  f{fi+1:<2}  {d:>3}  {y_best:>12.6f}  [{x_str}]  {ei_val:>10.6f}')

print('\nGP-EI optimisation complete for all 8 functions.')

In [ ]:
# ============================================================
# CELL 5 — PER-FUNCTION STRATEGIC ANALYSIS (ROUNDS 1–9)
# ============================================================

print('=' * 70)
print('PER-FUNCTION ANALYSIS — ROUNDS 1-9 HISTORY')
print('=' * 70)

# ---- f1 ----------------------------------------------------
print()
print('f1 (d=2) — Micro-Gradient Exploitation (Near-Zero Landscape)')
print('  Outputs effectively zero until R5; consistent positive trend R7→R9.')
for i in [6, 7, 8]:
    print(f'  R{i+1}: {func_X[0][i]} → {func_y[0][i]:.4e}')
print('  Trend: x1 +0.002/round, x2 -0.002/round → monotone improvement.')
print('  Strategy R10: x1=0.481, x2=0.497  (continue gradient direction)')

# ---- f2 ----------------------------------------------------
print()
print('f2 (d=2) — Sharp Peak; R1 All-Time Best Unbeaten')
print(f'  Best: R1 [0.695196, 0.395970] → {func_y[1].max():.6f}')
print('  R9 regressed to 0.367 after R8=0.574. Peak very narrow.')
print('  Strategy R10: [0.696000, 0.395500] — tighter bracket around R1')

# ---- f3 ----------------------------------------------------
print()
print('f3 (d=3) — R7 Confirmed Optimum; R8/R9 Regressed')
for i in [6, 7, 8]:
    x = func_X[2][i]; y_ = func_y[2][i]
    print(f'  R{i+1}: [{x[0]:.3f}, {x[1]:.3f}, {x[2]:.3f}] → {y_:.6f}')
print('  R7 = local optimum. R8 regressed (x1↓/x3↑). R9 continued worse.')
print('  Strategy R10: nudge around R7 → [0.476, 0.225, 0.410]')

# ---- f4 ----------------------------------------------------
print()
print('f4 (d=4) — R8 Best (0.4711); R9 Slight Drop → Stabilise')
for i in [6, 7, 8]:
    x = func_X[3][i]; y_ = func_y[3][i]
    print(f'  R{i+1}: [{x[0]:.3f},{x[1]:.3f},{x[2]:.3f},{x[3]:.3f}] → {y_:.6f}')
print('  R9 move (x1↓, x2↑) regressed from R8 best. Revert toward R8.')
print('  Strategy R10: [0.430, 0.430, 0.376, 0.399]')

# ---- f5 ----------------------------------------------------
print()
print('f5 (d=4) — R6 x1=0.005 gave highest 4440.4830; probe x1=0.003')
for i in [4, 5, 6, 7, 8]:
    x = func_X[4][i]; y_ = func_y[4][i]
    print(f'  R{i+1}: x1={x[0]:.4f} → {y_:.6f}')
print('  R6 x1=0.005 > all x1=0.000 rounds. Non-monotone near x1=0.')
print('  Strategy R10: x1=0.003 → [0.003000, 0.999999, 0.999999, 0.999999]')

# ---- f6 ----------------------------------------------------
print()
print('f6 (d=5) — R1 Best (-0.5508); 8 probes have not beaten it')
for i in [0, 7, 8]:
    x = func_X[5][i]; y_ = func_y[5][i]
    print(f'  R{i+1}: [{x[0]:.4f},{x[1]:.4f},{x[2]:.4f},{x[3]:.4f},{x[4]:.4f}] → {y_:.6f}')
print('  x4=0.999999, x5=0.000 are structural. Tighten x1→0.466, x2/x3 fine-probe.')
print('  Strategy R10: [0.466000, 0.242000, 0.575000, 0.999999, 0.000000]')

# ---- f7 ----------------------------------------------------
print()
print('f7 (d=6) — 5-Round Monotone Gain; Strongest Active Gradient')
deltas = []
for i in [4, 5, 6, 7, 8]:
    x = func_X[6][i]; y_ = func_y[6][i]
    coords = ', '.join([f'{v:.3f}' for v in x])
    print(f'  R{i+1}: [{coords}] → {y_:.6f}')
    if i > 4:
        deltas.append(func_y[6][i] - func_y[6][i-1])
print(f'  Δ per round: {[f"+{d:.4f}" for d in deltas]}')
print('  Strategy R10: uniform -0.003 step → [0.000, 0.226, 0.310, 0.200, 0.355, 0.730]')

# ---- f8 ----------------------------------------------------
print()
print('f8 (d=8) — x3 Monotone Gradient; All Other Dims Locked')
for i in [6, 7, 8]:
    x = func_X[7][i]; y_ = func_y[7][i]
    print(f'  R{i+1}: x3={x[2]:.3f} → {y_:.10f}')
delta_f8 = func_y[7][8] - func_y[7][7]
print(f'  Δ R8→R9: +{delta_f8:.10f}  (decelerating; x3 +0.002/round)')
print('  Strategy R10: x3=0.128→0.130. All other dims frozen.')

In [ ]:
# ============================================================
# CELL 6 — ROUND 10 FINAL QUERIES (STRATEGIC + GP-EI INFORMED)
# ============================================================
# Final queries integrate GP-EI output with directional analysis.
# Where GP-EI agrees with strategic direction → follow GP-EI.
# Where strategic signal is stronger (plateau/monotone) → override.

round10_queries = [
    # f1: Continue micro-gradient — x1↑0.002, x2↓0.002 (5-round trend)
    np.array([0.481000, 0.497000]),

    # f2: Tighter bracket around R1 best — x1↑0.001, x2↓0.0005
    np.array([0.696000, 0.395500]),

    # f3: Nudge around R7 optimum — x1↓0.002, x2↑0.002, x3↑0.002
    np.array([0.476000, 0.225000, 0.410000]),

    # f4: Stabilise near R8 best — revert x1↑, x2↓ from R9 regression
    np.array([0.430000, 0.430000, 0.376000, 0.399000]),

    # f5: Probe x1=0.003 — R6 x1=0.005 gave highest ever; midpoint test
    np.array([0.003000, 0.999999, 0.999999, 0.999999]),

    # f6: Tight exploit R1 zone — x1=0.466, x2/x3 fine-probe, x4/x5 held
    np.array([0.466000, 0.242000, 0.575000, 0.999999, 0.000000]),

    # f7: Uniform -0.003 step from R9 — 5-round monotone gradient confirmed
    np.array([0.000000, 0.226000, 0.310000, 0.200000, 0.355000, 0.730000]),

    # f8: x3 gradient continuation — 0.128→0.130; all other dims frozen
    np.array([0.064016, 0.008062, 0.130000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010]),
]

# Validation: dimension and bounds checks
for fi, x in enumerate(round10_queries):
    assert x.shape[0] == dims[fi], f'Dimension mismatch for f{fi+1}'
    assert np.all(x >= DOMAIN_LOWER) and np.all(x <= DOMAIN_UPPER), \
        f'Bounds violation for f{fi+1}: {x}'

print('All dimension and bounds checks passed.\n')

# Compare GP-EI suggestion vs strategic final
print(f'{"Func":>5}  {"d":>3}  {"GP-EI Suggestion":>50}  {"Strategic Final":>50}')
print('-' * 115)
for fi in range(N_FUNCTIONS):
    gp_str  = '-'.join([f'{v:.6f}' for v in gp_suggestions[fi]])
    fin_str = '-'.join([f'{v:.6f}' for v in round10_queries[fi]])
    print(f'  f{fi+1:<2}  {dims[fi]:>3}  {gp_str:>50}  {fin_str:>50}')

In [ ]:
# ============================================================
# CELL 7 — GP POSTERIOR DIAGNOSTICS AT ROUND 10 QUERIES
# ============================================================

print('GP Posterior at Round 10 Query Points')
print('=' * 90)
print(f'{"Func":>5}  {"y_best (obs)":>14}  {"μ (pred)":>12}  '
      f'{"σ (pred)":>12}  {"EI":>10}  {"Mode":>22}')
print('-' * 90)

mode_labels = [
    'Micro-gradient x1↑x2↓',   # f1
    'Bracket R1 peak',           # f2
    'Nudge R7 optimum',          # f3
    'Revert R8 direction',       # f4
    'Probe x1=0.003',            # f5
    'Tight exploit R1',          # f6
    'Uniform -0.003 step',       # f7
    'x3 +0.002 gradient',        # f8
]

for fi in range(N_FUNCTIONS):
    X      = func_X[fi]
    y      = func_y[fi]
    y_best = np.max(y)
    gp, scaler = fit_gp(X, y)

    x_q    = round10_queries[fi]
    x_sc   = scaler.transform(x_q.reshape(1, -1))
    mu, sg = gp.predict(x_sc, return_std=True)
    mu, sg = float(mu[0]), float(sg[0])

    ei_val = -expected_improvement(x_q, gp, scaler, y_best)

    print(f'  f{fi+1:<2}  {y_best:>14.6f}  {mu:>12.6f}  '
          f'{sg:>12.6f}  {ei_val:>10.6f}  {mode_labels[fi]:>22}')

print()
print('Note: High σ at query point = exploration value.')
print('      Low σ, high μ         = pure exploitation.')

In [ ]:
# ============================================================
# CELL 8 — ROUND 10 SUBMISSION STRINGS (PORTAL FORMAT)
# ============================================================
# Format: x1-x2-...-xn
# Each xi: begins with 0, six decimal places.

print('=' * 70)
print('ROUND 10 — FINAL SUBMISSION STRINGS (W21 / Module 21.1)')
print('=' * 70)
print()

func_labels = [
    'F1 (d=2)', 'F2 (d=2)', 'F3 (d=3)', 'F4 (d=4)',
    'F5 (d=4)', 'F6 (d=5)', 'F7 (d=6)', 'F8 (d=8)'
]

strategies = [
    'Micro-gradient nudge: x1↑0.002, x2↓0.002 (R7→R9 confirmed trend)',
    'Bracket R1 peak: x1↑0.001 above R1, x2↓0.0005 — tight exploit',
    'Nudge R7 optimum: x1↓0.002, x2↑0.002, x3↑0.002 (R8/R9 regressed)',
    'Stabilise near R8 best: revert x1↑, x2↓ from R9 regression',
    'Probe x1=0.003 — R6 x1=0.005 gave highest 4440.4830; midpoint EI',
    'Tight exploit R1 zone: x1=0.466 (+0.001), x2/x3 fine-probe',
    'Uniform -0.003 step from R9 — 5-round monotone gradient (R5→R9)',
    'x3 gradient: 0.128→0.130 (+0.002); all 7 other dims locked',
]

submission_strings = []
for fi in range(N_FUNCTIONS):
    s = '-'.join([f'{v:.6f}' for v in round10_queries[fi]])
    submission_strings.append(s)
    print(f'{func_labels[fi]}:  {s}')
    print(f'    Strategy : {strategies[fi]}')
    print(f'    y_best (R1-R9): {func_y[fi].max():.8f}')
    print()

print()
print('--- COPY-PASTE BLOCK ---')
for fi in range(N_FUNCTIONS):
    print(submission_strings[fi])

In [ ]:
# ============================================================
# CELL 9 — CUMULATIVE BEST TRACKER (ALL 9 ROUNDS)
# ============================================================

print('Cumulative Best Values by Round')
print('=' * 110)
header = f'{"Round":>7}' + ''.join([f'  {"f"+str(i+1):>12}' for i in range(N_FUNCTIONS)])
print(header)
print('-' * 110)

best_so_far = [-np.inf] * N_FUNCTIONS

for rnd in range(N_ROUNDS):
    row = f'  R{rnd+1:>4}'
    for fi in range(N_FUNCTIONS):
        y_val = all_outputs[rnd][fi]
        if y_val > best_so_far[fi]:
            best_so_far[fi] = y_val
        row += f'  {best_so_far[fi]:>12.4f}'
    print(row)

print('-' * 110)
final_row = '  Final'
for fi in range(N_FUNCTIONS):
    final_row += f'  {best_so_far[fi]:>12.4f}'
print(final_row)

print()
print('Key observations after 9 rounds:')
print('  f1 — Near-zero; micro-gradient trend +2e-7/round confirmed R7→R9')
print('  f2 — R1 best (0.7237) unbeaten; R9 regressed — peak is sharp/narrow')
print('  f3 — R7 optimum (-0.0353) confirmed; R8+R9 both regressed')
print('  f4 — R8 best (0.4711); R9 slight drop — plateau vicinity reached')
print('  f5 — R6 x1=0.005 gave 4440.4830 > all x1=0 rounds; x1 non-trivial')
print('  f6 — R1 best (-0.5508) after 9 rounds; structural x4=1, x5=0 confirmed')
print('  f7 — Strongest gradient: 2.207→2.271 over R1→R9; -0.003/round law')
print('  f8 — x3 sole active gradient: 9.85958→9.85967; +0.002/round')

In [ ]:
# ============================================================
# CELL 10 — TRANSPARENCY & INTERPRETABILITY REFLECTION
# ============================================================
# Required capstone component 21.1 — Module theme integration

reflection = """
REFLECTION: Transparency and Interpretability in BBO (Round 10)
================================================================

This round applies Module 21's core principle: a query's value lies
not only in its expected output but in the clarity of its rationale.

INTERPRETABILITY OF THE SEARCH STRATEGY
----------------------------------------
Each of the 8 decisions above is grounded in a specific, falsifiable
claim about the landscape:

  f1 : Micro-gradient claim  — outputs grow +0.002/round along x1↑,x2↓
  f2 : Sharp-peak claim      — R1 is a global maximum; nearby points degrade
  f3 : Local-optimum claim   — R7 is a saddle/peak; R8+R9 both confirmed regression
  f4 : Plateau claim         — system is near max; small perturbation to confirm
  f5 : Non-monotone claim    — x1=0.005 beats x1=0.000; optimum at x1∈(0,0.005)
  f6 : Boundary-anchor claim — x4=0.999999, x5=0.000 are structural constraints
  f7 : Linear-drift claim    — uniform -0.003/round law, 5 consecutive confirmations
  f8 : Single-variable claim — x3 is the sole remaining active gradient

TRANSPARENCY OF THE GP MODEL
------------------------------
The GP posterior (Cell 7) makes the model's uncertainty explicit at
each proposed query point:
  - Low σ + high μ  → pure exploitation (f7, f8)
  - Moderate σ      → EI-driven explore-exploit balance (f1, f3, f4)
  - High σ          → deliberate probe into uncertain region (f5)

This is the BBO analogue of a model card: the 'why' is as auditable
as the 'what'. A peer reviewer could reconstruct the same query from
the data alone, without access to the black-box evaluator.

REPRODUCIBILITY
---------------
  Random seed  : 42 (fixed globally)
  GP restarts  : 35 (acquisition) + 10 (kernel hyperparameters)
  Warm-start   : top-35 of 5,000 random candidates → L-BFGS-B
  Kernel       : Matérn-5/2 ARD + ConstantKernel + WhiteKernel
  Scaler       : StandardScaler (zero-mean, unit-variance)
  All code, data, and rationale are version-controlled on GitHub.
"""

print(reflection)